# EAGLE-3 draft for granite-docling

The Medusa route works but loses: `medusa` is in neither `EagleModelTypes` nor the V2-runner
allowlist, so vLLM disables async scheduling *and* falls back to the V1 model runner. Measured, a
drafted round costs 3.33 ms against plain vLLM's 1.20 ms, and break-even lands at 2.78 tokens/step
where the head delivers 1.20 pooled.

`eagle3` keeps both. Measured here with a random-init drafter (acceptance 0, so tok/s *is* the
round cost):

| config | round | drafter adds | break-even |
|---|---:|---:|---:|
| plain vLLM | 1.352 ms | — | 1.000 |
| eagle3 k=1 | 2.147 ms | 0.795 ms | **1.588** |
| eagle3 k=2 | 2.333 ms | 0.981 ms | **1.726** |
| eagle3 k=3 | 2.442 ms | 1.090 ms | 1.807 |
| eagle3 k=5 | 2.970 ms | 1.618 ms | 2.197 |

So eagle3 roughly halves Medusa's bar (2.78 → 1.59 at k=1) but it is **not** the ~1.1 that a
free-drafter calculation suggests: the drafter has a ~0.68 ms fixed cost plus ~0.11 ms per extra
step, and that is paid on every round. Low k is favoured — the marginal token is cheap but the
first one is not.

**What this model is.** Not the latent draft ported over. EAGLE-3 is autoregressive: propose a
token, embed it, feed it back through the drafter's own KV cache, repeat. `Eagle3LlamaForCausalLM`
returns one hidden state per position, so the `[B, T, HORIZON, 576]` parallel-head contract does
not survive — `future_heads` has nowhere to go. What *does* survive is the window: the drafter
attends over its own KV cache, so history comes back without being passed in.

**Prerequisite: traces with EAGLE-3 taps.** The V2 runner sets `use_aux_hidden_state_outputs=True`
unconditionally for `method="eagle3"`, so the target emits layers 2/14/27 concatenated and the
drafter's `fc` maps 1728 → 576. The current corpus stores only `last_hidden_state`, so it must be
re-extracted (4x larger on disk):

```
uv run fastdocling-extract data/images data/traces_taps --keep-taps
```


In [ ]:
from pathlib import Path
from time import perf_counter

import numpy as np
import torch
from torch.nn import functional as F
from torchinfo import summary
from tqdm.auto import tqdm

from fastdocling.data import (corpus_key, ensure_packed, output_vocab, packed_batches, plan,
                              prefetch_to_device, scan_traces, source_counts, split)
from fastdocling.eagle3 import AUX_LAYERS, Eagle3Draft, export_eagle3_checkpoint, init_from_target

# Paths resolve against the repo root, not the working directory: this notebook lives in
# notebooks/eagle3/, so a bare Path("data/traces") would point at notebooks/eagle3/data/traces.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())

# The corpora live on /mnt/ai; data/* are symlinks into it, so these paths work either way.
# Two roots, kept apart on disk because they are not the same thing:
#
#   traces_taps     labels are the target's *own* greedy DocTags -- on-policy, what EAGLE wants
#   traces_prefill  labels borrowed from DoclingMatix and teacher-forced.  The target agrees
#                   with them for 95.4% of label tokens (half the gap is <loc_N> coordinate
#                   digits, where the boxes come from Docling's layout model rather than from
#                   granite-docling).  Cheap -- one prefill per page instead of a full decode --
#                   but slightly off-policy supervision.
#
# Drop the second entry to train on the clean corpus alone; the cell below prints the mix.
TRACES = [ROOT / "data/traces_taps", ROOT / "data/traces_prefill"]
CACHE = ROOT / "data/cache"
PACK = CACHE / "pack_eagle3"         # built once from TRACES; see the packing cell below
OUTPUT = ROOT / "checkpoints/eagle3_draft.pt"
DECODES = CACHE / "decodes"

CONTEXT_LENGTH = 128     # training window; at inference the drafter's own KV cache carries history
BATCH_SIZE = 16
EPOCHS = 5
SPEC_TOKENS = 4         # num_speculative_tokens; k=1..2 is where break-even is lowest
MUON_LR, ADAMW_LR, WARMUP_STEPS = 0.02, 3e-4, 10
VLLM_GPU_FRACTION = 0.35
IN_DOMAIN_PAGES = 8     # holdout pages measured end to end; 3 lets a single page dominate
                        # the pooled ratio, which is how 1.20 tokens/step once read as 2.07
PRUNE_VOCAB = True      # d2t pruning; unlike Medusa's token_map this is not broken upstream

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- throughput ---------------------------------------------------------------------------
# This loop was data-bound, not compute-bound: re-deriving each batch from the safetensors cost
# 22 ms with the corpus in page cache and ~290 ms without, against ~20 ms of compute -- 52% to
# 91% of the step spent fetching.  Three changes, measured at BATCH_SIZE=48:
#
#   packed corpus + prefetch thread   data 22 ms -> 0.6 ms of observed wait
#   bfloat16 autocast, CE on bf16     compute 20.4 -> 11.0 ms   (fp32 logits are [3072, 30753],
#                                                                378 MB written per forward)
#   TF32 + torch.compile              compute 11.0 -> 9.1 ms
#
# Together 50.7 -> 10.2 ms/step end to end (3 epochs in 27 s), and the split inverts to ~6%
# data / ~94% compute.  Of what is left, Muon's Newton-Schulz is ~3 ms: lower MUON_NS_STEPS
# if that trade is worth it to you.
COMPILE = True          # compile the training step only; acceptance() runs the eager module
PREFETCH_DEPTH = 4      # batches kept in flight ahead of the GPU
AMP_DTYPE = torch.bfloat16
MUON_NS_STEPS = 5       # Newton-Schulz iterations; 3 measured 2.97 -> 2.02 ms per step

# Weights stay float32 (Muon's orthogonalisation and the optimizer state want the range); only
# the matmuls run in bfloat16.
torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision("high")
autocast = lambda: torch.autocast(device.type, dtype=AMP_DTYPE, enabled=device.type == "cuda")

In [ ]:
infos = scan_traces(TRACES, min_completion=CONTEXT_LENGTH + 2)
missing = [i for i in infos if not i.has_taps]
if missing:
    roots = sorted({i.source for i in missing})
    raise RuntimeError(
        f"{len(missing)}/{len(infos)} traces have no EAGLE-3 taps (in {', '.join(roots)}).  vLLM's "
        f"V2 runner always requests aux hidden states for method='eagle3', so the drafter's fc "
        f"needs layers {AUX_LAYERS} concatenated (1728-d).  Re-extract those roots with "
        f"--keep-taps.")

train_infos, holdout_infos = split(infos, holdout_fraction=0.02)
# first_offset has to match the loop's, or TOTAL_STEPS undercounts and the cosine schedule
# reaches zero before the last epoch does: at the default 2 this planned 881 steps/epoch
# against the 896 the loop actually runs, and the final 45 steps trained at lr 0.
schedule = plan(train_infos, CONTEXT_LENGTH, BATCH_SIZE, horizon=2, first_offset=1)
TOTAL_STEPS = schedule.steps_for_epochs(EPOCHS)
CORPUS_KEY = corpus_key(train_infos)
print(schedule.describe(steps=TOTAL_STEPS))

# What the corpus is actually made of.  Borrowed labels agree with the target for ~95% of tokens,
# so a run that is mostly traces_prefill is training mostly on that 95% -- worth seeing before
# reading an acceptance number, and worth checking again in the holdout, which is sampled from
# the same pool and so is only as clean as the mix.
mix = source_counts(infos)
total = sum(mix.values())
print("corpus: " + ", ".join(f"{k} {v:,} pages ({v / total:.0%})" for k, v in sorted(mix.items())))
print(f"holdout: {len(holdout_infos)} pages ({source_counts(holdout_infos)}); corpus key {CORPUS_KEY}")

# Keyed by CORPUS_KEY and kept in CACHE rather than in a trace root: the vocabulary is a property
# of the page *set*, not of any one corpus, and the set now spans two directories.
vocab = output_vocab(train_infos, cache=CACHE / f"output_vocab_{CORPUS_KEY}.npy") if PRUNE_VOCAB else None
if vocab is not None:
    print(f"draft vocabulary pruned to {len(vocab):,} of 100,352 tokens (written as d2t)")

# Pack the whole corpus -- train and holdout -- into one contiguous float16 array.  Rebuilt only
# when the page set changes (the guard is corpus_key), so re-running this cell is free.  Sizing:
# ~3.5 KB per position (1728-d float16), so the merged corpus is tens of GB -- which is why
# data/cache is a symlink onto /mnt/ai rather than the repo disk.
bar = tqdm(total=len(infos), unit="page", desc="packing", dynamic_ncols=True, leave=False)
pack = ensure_packed(infos, PACK, progress=lambda done, total: bar.update(done - bar.n))
bar.close()
print(f"pack: {pack.positions:,} positions at {pack.dim}-d in {pack.root}")

## Alignment

At cursor `i` the target's state `h_i` is known, and so is token `i+1` — it is the bonus token the
target already emitted. The drafter embeds that token, combines it with `h_i`, and must predict
token `i+2`. So the drafter's *input* token is at offset 1 and its *label* is at offset 2, which is
`FIRST_OFFSET`, the same convention the latent draft and the live loop use.

`packed_batches(..., horizon=2, first_offset=1)` yields exactly those two columns:
`tokens[:, :, 0]` is the input embedding's token, `tokens[:, :, 1]` is the label. It windows the
corpus the same way `iterate_batches` does — non-overlapping, from a per-page random offset that
moves with the seed — but reads them out of the pack instead of re-deriving them, and does not
build the target hidden states, which EAGLE-3 never trains on.

In [ ]:
draft = Eagle3Draft(
    in_dim=576 * len(AUX_LAYERS),
    draft_vocab_size=len(vocab) if vocab is not None else 100_352,
)

# The target's embedding and vocabulary projection are frozen, but they are not *arbitrary*: both
# have to be the target's own tensors.  vLLM binds the target's embed_tokens at load time, so a
# drafter trained against a random table learned out of an input space it will never see; and with
# a pruned vocabulary the checkpoint ships its own lm_head, so whatever sits there at export time
# is what gets served.  init_from_target fills both -- lm_head with the target's rows for `vocab`,
# in the order d2t maps back.  What is left to learn is fc + one decoder layer + norm.
init_from_target(draft, vocab=vocab)
draft = draft.to(device)

trainable = [p for n, p in draft.named_parameters() if not n.startswith(("embed_tokens", "lm_head"))]
for n, p in draft.named_parameters():
    if n.startswith(("embed_tokens", "lm_head")):
        p.requires_grad_(False)
print(f"{sum(p.numel() for p in trainable):,} trainable of {sum(p.numel() for p in draft.parameters()):,} total")

summary(draft, input_data=(torch.zeros(2, CONTEXT_LENGTH, 576 * len(AUX_LAYERS), device=device),
                           torch.zeros(2, CONTEXT_LENGTH, dtype=torch.long, device=device)),
        depth=3, col_names=("input_size", "output_size", "num_params"), row_settings=("var_names",))

In [ ]:
muon_params = [p for p in trainable if p.ndim == 2]
adamw_params = [p for p in trainable if p.ndim != 2]
optimizers = [torch.optim.Muon(muon_params, lr=MUON_LR, weight_decay=0.01, momentum=0.95,
                               nesterov=True, ns_steps=MUON_NS_STEPS)]
if adamw_params:
    optimizers.append(torch.optim.AdamW(adamw_params, lr=ADAMW_LR, weight_decay=0.01))
warmup_cosine = lambda s: min(1.0, (s + 1) / WARMUP_STEPS) * 0.5 * (1 + np.cos(np.pi * min(1.0, s / max(1, TOTAL_STEPS))))
lr_schedules = [torch.optim.lr_scheduler.LambdaLR(o, warmup_cosine) for o in optimizers]

# Map target token ids onto draft rows when the head is pruned; ids outside the set are ignored.
if vocab is not None:
    target_to_draft = torch.full((100_352,), -100, dtype=torch.long, device=device)
    target_to_draft[torch.as_tensor(vocab, device=device)] = torch.arange(len(vocab), device=device)
else:
    target_to_draft = None

# torch.compile guards on shape, so the training loop gets the compiled module and acceptance()
# -- which ends on a short final batch -- gets the eager one, rather than paying a recompile.
model = torch.compile(draft) if COMPILE and device.type == "cuda" else draft


def batches(infos, seed=0, drop_last=True):
    """(aux_states [B,T,1728], input token [B,T], label [B,T]) -- see the alignment note.

    Windows come from the pack as float16 and are cast on the device: half the bytes over PCIe
    and no float32 upcast on the CPU.  The prefetch thread fills the next batch while the GPU is
    still on this one, which is what keeps the fetch off the critical path.
    """
    # pin=True fills the batch straight into pinned memory, so the H2D copy is async and the
    # prefetch thread does not pay a second host-to-host copy to get there (~3.3 ms/batch).
    stream = packed_batches(pack, infos, CONTEXT_LENGTH, BATCH_SIZE, horizon=2, seed=seed,
                            first_offset=1, drop_last=drop_last, pin=device.type == "cuda")
    for x, tok in prefetch_to_device(stream, device, depth=PREFETCH_DEPTH):
        yield x.to(AMP_DTYPE), tok[:, :, 0], tok[:, :, 1]


@torch.inference_mode()
def acceptance(infos):
    """Teacher-forced top-1 accuracy of the drafter's next-token prediction.

    Labels outside the pruned vocabulary are masked out rather than counted as misses.  At
    serve time the head cannot propose them at all, so they are guaranteed rejections and this
    reads very slightly high -- 182 of 57,673 holdout positions, 0.3%, on this corpus.
    """
    draft.eval(); hits = total = 0
    for x, inp, label in batches(infos, seed=0, drop_last=False):
        with autocast():
            pred = draft(x, inp).argmax(-1)
        gold = target_to_draft[label] if target_to_draft is not None else label
        mask = gold >= 0
        hits += (pred[mask] == gold[mask]).sum().item(); total += int(mask.sum())
    draft.train()
    return hits / max(1, total)


history, step, started = [], 0, perf_counter()
progress = tqdm(total=TOTAL_STEPS, unit="step", dynamic_ncols=True)
for epoch in range(EPOCHS):
    for x, inp, label in batches(train_infos, seed=epoch):
        gold = target_to_draft[label] if target_to_draft is not None else label
        with autocast():
            logits = model(x, inp)
        # Deliberately outside autocast, which would put cross_entropy back in float32: the
        # logits are [B*T, draft_vocab] = [3072, 30753], and materialising those in float32 cost
        # more than the projection that produced them (378 MB per forward against 189 MB).  The
        # softmax still accumulates in float32, so the loss agrees with an all-float32 step to
        # ~1e-3 relative and the gradients to a cosine of 0.999994 -- but note loss.item() is
        # bfloat16-quantised, so the printed value carries about three significant digits.
        loss = F.cross_entropy(logits.flatten(0, 1), gold.flatten(), ignore_index=-100)
        for o in optimizers:
            o.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(trainable, 1.0)
        for o, s in zip(optimizers, lr_schedules):
            o.step(); s.step()
        step += 1; progress.update()
        if step % 200 == 0:
            history.append((step, loss.item()))
            progress.set_postfix(loss=f"{loss.item():.3f}", lr=f"{lr_schedules[0].get_last_lr()[0]:.1e}")
progress.close()
elapsed = perf_counter() - started
print(f"{step:,} steps in {elapsed:.0f}s = {elapsed / max(1, step) * 1e3:.2f} ms/step "
      f"({schedule.tokens_per_step * step / elapsed:,.0f} tokens/s)")

p1 = acceptance(holdout_infos)
print(f"\nholdout next-token accuracy {p1:.3f}  ->  k=1 gives {1 + p1:.3f} tokens/step "
      f"(break-even 1.588), k=2 at best {1 + p1 + p1 ** 2:.3f} (break-even 1.726)")
OUTPUT.parent.mkdir(exist_ok=True)
torch.save({"state_dict": draft.state_dict(), "accuracy": p1, "aux_layers": list(AUX_LAYERS)}, OUTPUT)

## Measure it in vLLM

Export writes vLLM's on-disk EAGLE-3 layout (`layers.0.*`, `fc`, `norm`, optional `d2t`) and omits
`embed_tokens`/`lm_head` so the engine binds the target's own. Report **pooled** aggregates — total
tokens over total seconds, total tokens over total rounds. Per-page means were distorted by a
factor of two on this page set, because one short page with high acceptance dominates an unweighted
average.

In [ ]:
from fastdocling.decode import available
from fastdocling.decode.vllm_decoder import sweep_speculative


def pooled(rows):
    """(tokens, tok/s, tokens per target step) summed over pages before dividing.

    Totals first, ratio second.  An unweighted mean of per-page rates lets a short page outvote
    a long one: on this corpus a 729-token page at 3.14 tokens/step and a 7050-token page at
    1.09 averaged to 2.07, against a pooled 1.20.
    """
    tok = sum(r["tokens"] for r in rows)
    sec = sum(r["tokens"] / r["decode_tps"] for r in rows)
    rounds = sum(r["tokens"] / r["tokens_per_round"] for r in rows)
    return tok, tok / sec, tok / rounds


def report(rows, title):
    """Pooled throughput per configuration, then what the drafter actually got right."""
    grouped = {}
    for r in rows:
        grouped.setdefault(r["config"], []).append(r)

    n_pages = len(next(iter(grouped.values())))
    print(f"\n=== {title}: {n_pages} pages, {pooled(grouped['baseline'])[0]:,} tokens ===")
    print(f"{'configuration':22s}{'tok/s (pooled)':>16s}{'tokens/step':>13s}{'vs plain':>10s}")
    plain = None
    for label, rs in grouped.items():
        _, tps, per_step = pooled(rs)
        plain = plain or tps
        print(f"{label:22s}{tps:16.1f}{per_step:13.3f}{tps / plain:9.2f}x")

    # accepted_per_pos[i] counts rounds whose proposal at slot i was accepted.  Slot i is only
    # reached when slot i-1 was accepted, so its denominator is the acceptances at slot i-1 (and
    # for slot 0, the number of drafting rounds).  A flat accepted/proposed hides that the later
    # slots are judged on a much smaller population -- and it is the later slots that decide
    # whether raising num_speculative_tokens pays, since break-even rises with k.
    spec_rows = [r for r in rows if r["spec"] is not None]
    if not spec_rows:
        return grouped
    width = max(len(r["accepted_per_pos"]) for r in spec_rows)
    per_pos = [sum(r["accepted_per_pos"][i] if i < len(r["accepted_per_pos"]) else 0
                   for r in spec_rows) for i in range(width)]
    rounds_drafted = sum(r["drafts"] for r in spec_rows)
    proposed = sum(r["draft_tokens"] for r in spec_rows)
    print(f"\ncorrectly drafted: {rounds_drafted:,} drafting rounds, {proposed:,} tokens proposed")
    print(f"{'slot':>6s}{'reached':>10s}{'accepted':>10s}{'rate':>8s}")
    reached = rounds_drafted
    for i, acc in enumerate(per_pos, 1):
        print(f"{i:>6d}{reached:>10,}{acc:>10,}{acc / max(1, reached):>8.1%}")
        reached = acc
    print(f"{'all':>6s}{proposed:>10,}{sum(per_pos):>10,}{sum(per_pos) / max(1, proposed):>8.1%}")
    return grouped


if "vllm" not in available():
    print("vLLM is not installed here (`uv sync --extra cuda`); skipping.")
else:
    # share_embeddings=True always: vLLM binds the target's frozen embed_tokens, and decides
    # that independently of lm_head, which a pruned vocabulary still has to ship.  Tying the two
    # together bundled a 57.8M-parameter embedding the drafter never needed.
    ckpt_dir = export_eagle3_checkpoint(
        draft, CACHE / "eagle3_vllm", vocab=vocab, share_embeddings=True,
        aux_layers=AUX_LAYERS)
    print(f"exported {ckpt_dir}")

    def image_for(info):
        """Map a trace back to the page image it was extracted from.

        The two corpora name pages differently because they arrived differently: a rendered PDF
        keeps its directory structure in the name (ml_papers__<doc>__0001), while a borrowed page
        is flat and sits next to its manifest under data/prefill/<tag>/images.  Extension is
        whatever the source dataset stored, so glob for it rather than assuming .png.
        """
        if info.source == "traces_prefill":
            tag = info.path.stem.split("__")[0]
            hit = next((ROOT / "data/prefill" / tag / "images").glob(info.path.stem + ".*"), None)
            if hit is not None:
                return hit
        return ROOT.joinpath("data/images", *info.path.stem.split("__")).with_suffix(".png")

    SPEC_CFG = {"method": "eagle3", "model": str(ckpt_dir), "num_speculative_tokens": SPEC_TOKENS}
    rows = sweep_speculative([image_for(i) for i in holdout_infos[:IN_DOMAIN_PAGES]],
                             [None, SPEC_CFG], cache_dir=DECODES, batched=False,
                             gpu_memory_utilization=VLLM_GPU_FRACTION)
    report(rows, "in-domain holdout (ML papers)")

## Out of domain

The draft is trained on ML papers; Docling's real workload is broader. The same measurement on
`data/ood/images` -- tax forms, court opinions, standards, central-bank minutes, lecture notes --
answers the question the in-domain number cannot: whether the acceptance survives contact with
documents the draft has never seen.

Read the per-category table, not just the aggregate. Acceptance and wall-clock can move in
*opposite* directions here: a drafter pays its per-round cost on every round, so a long-output
document can accept more tokens per step and still finish slower than plain decoding.

In [ ]:
OOD_IMAGES = ROOT / "data/ood/images"
OOD_PAGES_PER_DOC = 2      # per document; 8 categories x ~1-2 docs keeps this to ~20 pages

if "vllm" not in available():
    print("vLLM is not installed here (`uv sync --extra cuda`); skipping.")
elif not OOD_IMAGES.is_dir():
    print("no OOD corpus: run `uv run python scripts/fetch_ood.py` then\n"
          "`uv run fastdocling-prep render data/ood/docs data/ood/images`")
else:
    ood_pages = [(cat.name, page)
                 for cat in sorted(p for p in OOD_IMAGES.iterdir() if p.is_dir())
                 for doc in sorted(p for p in cat.iterdir() if p.is_dir())
                 for page in sorted(doc.glob("*.png"))[:OOD_PAGES_PER_DOC]]
    category_of = {str(page): cat for cat, page in ood_pages}

    ood_rows = sweep_speculative([page for _, page in ood_pages], [None, SPEC_CFG],
                                 cache_dir=DECODES, batched=False,
                                 gpu_memory_utilization=VLLM_GPU_FRACTION)
    report(ood_rows, f"out of domain ({len({c for c, _ in ood_pages})} categories)")

    # Per category, because the aggregate hides the spread: long-output documents (forms) pay the
    # drafter's per-round cost over many more rounds than short ones, so a drafter that breaks
    # even on the mean can still lose badly on the documents that take the longest to decode.
    by_category = {}
    for r in ood_rows:
        by_category.setdefault(category_of[str(r["page"])], {}).setdefault(r["config"], []).append(r)
    print(f"\n{'category':16s}{'pages':>6s}{'tokens':>9s}{'plain':>9s}{'draft':>9s}{'vs plain':>10s}{'tok/step':>10s}")
    for cat in sorted(by_category):
        arms = by_category[cat]
        spec_label = next(k for k in arms if k != "baseline")
        tok, base_tps, _ = pooled(arms["baseline"])
        _, spec_tps, per_step = pooled(arms[spec_label])
        print(f"{cat:16s}{len(arms['baseline']):>6d}{tok:>9,}{base_tps:>9.1f}{spec_tps:>9.1f}"
              f"{spec_tps / base_tps:>9.2f}x{per_step:>10.3f}")